### Document Indexing with PyPDF

In [1]:
import copy

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter

/tmp/ipykernel_303/4098098794.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
loader_pdf = PyPDFLoader("files/Introduction_to_Data_and_Data_Science.pdf")
pages_pdf = loader_pdf.load()
page_num = len(pages_pdf)  # 6 pages
page_info = pages_pdf[4].metadata

In [3]:
# Remove \n new line characters as they consume tokens
pages = copy.deepcopy(pages_pdf)
pdf_page_1 = pages[0].page_content.split() # Makes a list and removes the \n chars
pdf_page_1 = " ".join(pdf_page_1)
pdf_page_1

'Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individually and examine how they relate to other parts. And that’s analysis in a nutshell. One important thing to remember, however, is that you perform analyses on things that have already happened in the past. Such as using an analysis to explain how a

Checking tokenizer count in open AI, the first page is 421 with \n and without \n, it's 306 tokens.. a huge reduction.. think of all six pages.

In [4]:
for page in pages:
    page.page_content = " ".join(page.page_content.split())  # Actually overwrites pages_pdf content

In [5]:
pages

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-11-09T10:16:34+02:00', 'author': 'Hristina  Hristova', 'moddate': '2023-11-09T10:16:34+02:00', 'source': 'files/Introduction_to_Data_and_Data_Science.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and 

In [22]:
char_splitter = CharacterTextSplitter(separator=".", chunk_size=500, chunk_overlap=0)
pages_split = char_splitter.split_documents(pages)
pages_split[1].page_content

'So, let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individually and examine how they relate to other parts. And that’s analysis in a nutshell'

### Indexing with DocLoader and MarkDown Splitter

In [31]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters.markdown import MarkdownHeaderTextSplitter

loader_docx = Docx2txtLoader("files/Introduction_to_Data_and_Data_Science.docx")
pages = loader_docx.load()

split_on = [("#", "Course Title"), ("##", "Lecture Title")]
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=split_on)
# Length is 1 page according to the loader, so we fetch it and split
pages_split = md_splitter.split_text((pages[0]).page_content)
print(pages_split[0].metadata, "\n", pages_split[1].metadata)

{'Course Title': 'Introduction to Splitting', 'Lecture Title': 'Analysis vs Analytics Alright!'} 
 {'Course Title': 'Introduction to Splitting', 'Lecture Title': 'Programming Languages & Software Employed in Data Science -'}


### Text Embedding with OpenAI

In [36]:
pages = copy.deepcopy(pages_split)
for num in range(len(pages)):
    pages[num].page_content = " ".join(pages[num].page_content.split())

pages_char_split = char_splitter.split_documents(pages)
pages_char_split

[Document(metadata={'Course Title': 'Introduction to Splitting', 'Lecture Title': 'Analysis vs Analytics Alright!'}, page_content='So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis'),
 Document(metadata={'Course Title': 'Introduction to Splitting', 'Lecture Title': 'Analysis vs Analytics Alright!'}, page_content='Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individually and examin

In [40]:
from langchain_openai.embeddings import OpenAIEmbeddings

from config import OPEN_AI_KEY as API_KEY

embedding = OpenAIEmbeddings(model="text-embedding-3-small", api_key=API_KEY)
vector_1 = embedding.embed_query(pages_char_split[3].page_content)
vector_2 = embedding.embed_query(pages_char_split[5].page_content)
vector_3 = embedding.embed_query(pages_char_split[18].page_content)

In [49]:
import numpy as np
# Compare how close they are to each other
print(np.dot(vector_1, vector_2), np.dot(vector_1, vector_3), np.dot(vector_2, vector_3))
print(np.linalg.norm(vector_1), np.linalg.norm(vector_2), np.linalg.norm(vector_3))

0.5469643853602975 0.3994167640659043 0.34540798791346106
1.0000864483738017 0.9999574364400718 0.9997614304560071
